# Pre-processing Pipeline

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import joblib
import re
import zlib

from pathlib import Path
from collections import Counter
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.model_selection import GroupShuffleSplit, train_test_split

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Helper functions for data collection
from scripts.collect_helper import check_cluster, pull_logs, get_collection_stats

## Connect to K8s cluster & Get all pods

In [ ]:
check_cluster()

## Data Collection

Pull logs from Cilium dan Tetragon

In [ ]:
pull_logs(sessions=144, interval=600, stimulate=True)
get_collection_stats()

In [ ]:
with open("raw_logs/hubble.json") as f:
    hubble_malicious = [json.loads(line) for line in f if line.strip() and "malicious-containers-stratus" in line.lower()][:2]

print(f"Hubble malicious:")
print(json.dumps(hubble_malicious, indent=1))

In [ ]:
with open("raw_logs/tetragon.json") as f:
    tetragon_socket = [json.loads(line) for line in f if line.strip() and "malicious-containers-stratus" in line.lower()][:2]
print(f"\nTetragon socket:")
print(json.dumps(tetragon_socket, indent=1))

In [ ]:
"""Multi-class scenario labeling.

The pipeline stores:
  - scenario_label: detailed behavior/scenario class
  - label: alias for scenario_label for backward compatibility
"""
DEFAULT_LABEL_CONFIG = {
    "scenario_class_names": [
        "Botnet",
        "Miner",
        "Trojan-Agent",
        "Cloud Attack",
        "Data-Caching",
        "Database",
        "Media-Streaming",
        "Web-Serving",
    ],
    "rules": [
        {"pattern": "malicious-containers-mirai", "scenario_label": 0},
        {"pattern": "cache-worker", "scenario_label": 0},
        {"pattern": "malicious-containers-miner", "scenario_label": 1},
        {"pattern": "malicious-containers-coinminer", "scenario_label": 1},
        {"pattern": "malicious-containers-kinsing", "scenario_label": 1},
        {"pattern": "frontend", "scenario_label": 1},
        {"pattern": "malicious-containers-agent", "scenario_label": 2},
        {"pattern": "backend-api", "scenario_label": 2},
        {"pattern": "malicious-containers-stratus", "scenario_label": 3},
        {"pattern": "memcached", "scenario_label": 4},
        {"pattern": "flask-postgres", "scenario_label": 5},
        {"pattern": "wordpress-mysql", "scenario_label": 5},
        {"pattern": "mariadb", "scenario_label": 5},
        {"pattern": "postgres", "scenario_label": 5},
        {"pattern": "mysql", "scenario_label": 5},
        {"pattern": "sysbench", "scenario_label": 5},
        {"pattern": "media-streaming", "scenario_label": 6},
        {"pattern": "web-serving", "scenario_label": 7},
        {"pattern": "flask", "scenario_label": 7},
        {"pattern": "wordpress", "scenario_label": 7},
        {"pattern": "stimulator", "scenario_label": 7},
    ],
}

def load_label_config():
    candidates = [Path("label_mapping.json")]
    for path in candidates:
        if path.exists():
            with open(path) as f:
                return json.load(f), path
    return DEFAULT_LABEL_CONFIG, None

LABEL_CONFIG, LABEL_CONFIG_PATH = load_label_config()
SCENARIO_CLASS_NAMES = LABEL_CONFIG.get("scenario_class_names", LABEL_CONFIG.get("class_names", []))
CLASS_NAMES = SCENARIO_CLASS_NAMES
LABEL_NAMES = {i: name for i, name in enumerate(SCENARIO_CLASS_NAMES)}
LABEL_RULES = sorted(
    LABEL_CONFIG["rules"],
    key=lambda rule: len(rule["pattern"]),
    reverse=True,
)

def normalize_name(value):
    return (value or "").lower().replace("_", "-")

def get_label_info(pod_name):
    pod_lower = normalize_name(pod_name)
    for rule in LABEL_RULES:
        if normalize_name(rule["pattern"]) in pod_lower:
            return int(rule.get("scenario_label", rule.get("label")))
    return -1

def get_label(pod_name):
    return get_label_info(pod_name)

print(f"Label config: {LABEL_CONFIG_PATH or 'built-in fallback'}")
print(f"Scenario labels: {SCENARIO_CLASS_NAMES}")
print("Rules:")
for rule in LABEL_RULES:
    scenario = int(rule.get("scenario_label", rule.get("label")))
    print(f"  scenario {scenario}: {rule['pattern']}")


## 2. Parse Tetragon (Syscall) - Per Pod 5-grams

In [ ]:
def iter_json_records(filepath):
    """Read either NDJSON or a JSON array/object without loading broken lines."""
    with open(filepath) as f:
        try:
            data = json.load(f)
            if isinstance(data, list):
                yield from data
            else:
                yield data
            return
        except json.JSONDecodeError:
            f.seek(0)

        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError:
                continue

def session_id_from_path(path):
    match = re.search(r"_session_(\d+)\.json$", str(path))
    return int(match.group(1)) if match else 0

def tetragon_log_paths():
    session_paths = sorted(
        Path("raw_logs/sessions").glob("tetragon_session_*.json"),
        key=session_id_from_path,
    )
    return session_paths or [Path("raw_logs/1_tetragon.json")]

def parse_tetragon_file(filepath):
    records = []
    session_id = session_id_from_path(filepath)

    for event_index, entry in enumerate(iter_json_records(filepath)):
        try:
            event_type = None
            process = {}
            syscall = ""

            if "process_kprobe" in entry:
                event_type = "kprobe"
                kprobe = entry["process_kprobe"]
                process = kprobe.get("process", {})
                syscall = kprobe.get("function_name", "")
            elif "process_exec" in entry:
                event_type = "exec"
                process = entry["process_exec"].get("process", {})
                syscall = entry["process_exec"].get("function_name") or "__x64_sys_execve"
            elif "process_exit" in entry:
                event_type = "exit"
                process = entry["process_exit"].get("process", {})
                syscall = "__x64_sys_exit_group"
            else:
                continue

            pod = process.get("pod", {})
            container = process.get("container", {})
            pod_name = pod.get("name", "")
            namespace = pod.get("namespace", "")
            scenario_label = get_label_info(pod_name)
            label = scenario_label

            records.append({
                "session_id": session_id,
                "event_index": event_index,
                "event_type": event_type,
                "timestamp": entry.get("timestamp", ""),
                "pod_name": pod_name,
                "namespace": namespace,
                "container": container.get("name", ""),
                "binary": process.get("binary", ""),
                "syscall": syscall,
                "label": scenario_label,
                "scenario_label": scenario_label,
            })
        except Exception:
            continue

    return records

def parse_tetragon():
    records = []
    paths = tetragon_log_paths()
    for path in paths:
        records.extend(parse_tetragon_file(path))

    df = pd.DataFrame(records)
    unknown = df[df["label"] < 0].copy()
    if not unknown.empty:
        print(f"Skipped {len(unknown):,} unlabeled Tetragon events. Top unknown pods:")
        print(unknown.groupby(["namespace", "pod_name"]).size().sort_values(ascending=False).head(15))

    df = df[df["label"] >= 0].copy()
    print(f"Parsed {len(df):,} labeled Tetragon events from {len(paths)} file(s)")
    print(f"Labels: {df['label'].value_counts().sort_index().to_dict()}")
    print(f"Sessions: {df['session_id'].nunique()}")
    return df

tetragon_df = parse_tetragon()
tetragon_df.head()

In [ ]:
SYSCALL_MAP = {
    "__x64_sys_read": 0,
    "__x64_sys_write": 1,
    "__x64_sys_open": 2,
    "__x64_sys_close": 3,
    "__x64_sys_stat": 4,
    "__x64_sys_fstat": 5,
    "__x64_sys_mmap": 9,
    "__x64_sys_mprotect": 10,
    "__x64_sys_munmap": 11,
    "__x64_sys_brk": 12,
    "__x64_sys_rt_sigaction": 13,
    "__x64_sys_rt_sigreturn": 15,
    "__x64_sys_ioctl": 16,
    "__x64_sys_access": 21,
    "__x64_sys_sched_yield": 24,
    "__x64_sys_nanosleep": 35,
    "__x64_sys_socket": 41,
    "__x64_sys_connect": 42,
    "__x64_sys_accept": 43,
    "__x64_sys_sendto": 44,
    "__x64_sys_recvfrom": 45,
    "__x64_sys_clone": 56,
    "__x64_sys_fork": 57,
    "__x64_sys_execve": 59,
    "__x64_sys_exit": 60,
    "__x64_sys_exit_group": 231,
    "__x64_sys_wait4": 61,
    "__x64_sys_kill": 62,
    "__x64_sys_openat": 257,
    "__x64_sys_newfstatat": 262,
    "__x64_sys_epoll_pwait": 281,
}

def stable_syscall_number(syscall):
    if syscall in SYSCALL_MAP:
        return SYSCALL_MAP[syscall]
    if not syscall:
        return -1
    # Avoid Python's randomized hash(), which changes values between processes.
    return 1000 + (zlib.crc32(syscall.encode("utf-8")) % 1000)

tetragon_df["syscall_num"] = tetragon_df["syscall"].apply(stable_syscall_number)

unmapped = tetragon_df[~tetragon_df["syscall"].isin(SYSCALL_MAP)]["syscall"].value_counts()
if len(unmapped) > 0:
    print(f"Unmapped syscalls kept with stable ids: {unmapped.head(20).to_dict()}")

tetragon_df = tetragon_df[tetragon_df["syscall_num"] >= 0].copy()

def generate_ngrams(sequence, n=5):
    return [tuple(sequence[i:i+n]) for i in range(len(sequence) - n + 1)]

group_cols = ["session_id", "pod_name", "namespace", "label"]
syscall_seqs = (
    tetragon_df
    .sort_values(["session_id", "pod_name", "event_index"])
    .groupby(group_cols)["syscall_num"]
    .apply(list)
    .reset_index(name="sequence")
)
syscall_seqs["ngrams"] = syscall_seqs["sequence"].apply(lambda seq: generate_ngrams(seq, 5))

ngram_records = []
for _, row in syscall_seqs.iterrows():
    for ngram_index, ngram in enumerate(row["ngrams"]):
        ngram_records.append({
            "session_id": row["session_id"],
            "pod_name": row["pod_name"],
            "namespace": row["namespace"],
            "label": row["label"],
            "scenario_label": row["label"],
            "ngram_index": ngram_index,
            "n1": ngram[0],
            "n2": ngram[1],
            "n3": ngram[2],
            "n4": ngram[3],
            "n5": ngram[4],
        })

syscall_dataset = pd.DataFrame(ngram_records)

print(f"Syscall: {len(syscall_dataset):,} 5-grams")
print(f"Labels: {syscall_dataset['label'].value_counts().sort_index().to_dict()}")
print(f"Duplicate full rows: {syscall_dataset.duplicated().sum():,}")
syscall_dataset.head()

## 3. Parse Hubble (NetworkFlow) Features

In [ ]:
def hubble_log_paths():
    session_paths = sorted(
        Path("raw_logs/sessions").glob("hubble_session_*.json"),
        key=session_id_from_path,
    )
    return session_paths or [Path("raw_logs/hubble.json")]

def endpoint_info(endpoint):
    pod_name = endpoint.get("pod_name", "") or ""
    namespace = endpoint.get("namespace", "") or ""
    scenario_label = get_label_info(pod_name)
    return {
        "pod_name": pod_name,
        "namespace": namespace,
        "scenario_label": scenario_label,
    }

def choose_labeled_endpoint(flow):
    src = endpoint_info(flow.get("source", {}))
    dst = endpoint_info(flow.get("destination", {}))
    direction = flow.get("traffic_direction", "")

    src_labeled = src["scenario_label"] >= 0
    dst_labeled = dst["scenario_label"] >= 0

    # Prefer the endpoint represented by Hubble traffic_direction when it has a label.
    if direction == "EGRESS" and src_labeled:
        return src, "source"
    if direction == "INGRESS" and dst_labeled:
        return dst, "destination"

    # If the direction endpoint is unlabeled, keep the flow when the opposite endpoint is labeled.
    if src_labeled and not dst_labeled:
        return src, "source"
    if dst_labeled and not src_labeled:
        return dst, "destination"

    # For service-to-service flows where both endpoints are labeled, keep a deterministic perspective.
    if src_labeled and dst_labeled:
        if direction == "INGRESS":
            return dst, "destination"
        return src, "source"

    # Preserve one endpoint for unknown-flow diagnostics in parse_hubble().
    fallback = src if src["pod_name"] else dst
    return fallback, "unknown"

def parse_hubble_file(filepath):
    records = []
    session_id = session_id_from_path(filepath)

    for flow_index, entry in enumerate(iter_json_records(filepath)):
        try:
            flow = entry.get("flow", {})
            if not flow:
                continue

            endpoint, flow_perspective = choose_labeled_endpoint(flow)
            pod_name = endpoint["pod_name"]
            namespace = endpoint["namespace"]
            if not pod_name:
                continue

            scenario_label = endpoint["scenario_label"]
            label = scenario_label

            l4 = flow.get("l4", {})
            tcp = l4.get("TCP", {})
            udp = l4.get("UDP", {})
            flags = tcp.get("flags", {})
            dst_port = int(tcp.get("destination_port", udp.get("destination_port", 0)) or 0)
            src_port = int(tcp.get("source_port", udp.get("source_port", 0)) or 0)

            records.append({
                "session_id": session_id,
                "flow_index": flow_index,
                "pod_name": pod_name,
                "namespace": namespace,
                "label": scenario_label,
                "scenario_label": scenario_label,
                "traffic_dir": flow.get("traffic_direction", "UNKNOWN"),
                "flow_perspective": flow_perspective,
                "proto_TCP": int(bool(tcp)),
                "proto_UDP": int(bool(udp)),
                "proto_OTHER": int(not tcp and not udp),
                "dir_EGRESS": int(flow.get("traffic_direction") == "EGRESS"),
                "dir_INGRESS": int(flow.get("traffic_direction") == "INGRESS"),
                "verdict_FORWARDED": int(flow.get("verdict") == "FORWARDED"),
                "verdict_DROPPED": int(flow.get("verdict") == "DROPPED"),
                "verdict_TRACED": int(flow.get("verdict") == "TRACED"),
                "flag_SYN": int(flags.get("SYN", False)),
                "flag_ACK": int(flags.get("ACK", False)),
                "flag_FIN": int(flags.get("FIN", False)),
                "flag_RST": int(flags.get("RST", False)),
                "flag_PSH": int(flags.get("PSH", False)),
                "flag_URG": int(flags.get("URG", False)),
                "is_reply": int(flow.get("is_reply", False)),
                "src_port": src_port,
                "dst_port": dst_port,
                "is_well_known_port": int(0 < dst_port < 1024),
                "is_high_port": int(dst_port > 49152),
                "is_dns_port": int(dst_port == 53),
                "is_http_port": int(dst_port in [80, 443, 8080, 8443]),
                "is_db_port": int(dst_port in [3306, 5432, 6379, 11211]),
                "is_mining_port": int(dst_port in [3333, 4444, 5555, 7777, 8888, 9999, 14444, 45700]),
            })
        except Exception:
            continue

    return records

def parse_hubble():
    records = []
    paths = hubble_log_paths()
    for path in paths:
        records.extend(parse_hubble_file(path))

    df = pd.DataFrame(records)
    unknown = df[df["label"] < 0].copy()
    if not unknown.empty:
        print(f"Skipped {len(unknown):,} unlabeled Hubble flows. Top unknown pods:")
        print(unknown.groupby(["namespace", "pod_name"]).size().sort_values(ascending=False).head(15))

    df = df[df["label"] >= 0].copy()
    if not df.empty and "flow_perspective" in df.columns:
        print(f"Flow perspectives: {df['flow_perspective'].value_counts().to_dict()}")
    print(f"Parsed {len(df):,} labeled Hubble flows from {len(paths)} file(s)")
    print(f"Labels: {df['label'].value_counts().sort_index().to_dict()}")
    print(f"Sessions: {df['session_id'].nunique()}")
    return df

network_events = parse_hubble()
network_events.head()

## 4. Feature Selection (Network)

SelectKBest k=20 dengan chi-squared

In [ ]:
metadata_cols = [
    "session_id", "pod_name", "namespace", "label",
    "scenario_label", "flow_window",
]
count_cols = [
    "proto_TCP", "proto_UDP", "proto_OTHER",
    "dir_EGRESS", "dir_INGRESS",
    "verdict_FORWARDED", "verdict_DROPPED", "verdict_TRACED",
    "flag_SYN", "flag_ACK", "flag_FIN", "flag_RST", "flag_PSH", "flag_URG",
    "is_well_known_port", "is_high_port", "is_dns_port", "is_http_port",
    "is_db_port", "is_mining_port",
]

if network_events.empty:
    raise ValueError("No labeled Hubble events were parsed.")

# Window Hubble flows within each pod-session. Without this, grouping by only
# session_id/pod_name collapses all flows from a pod-session into one row.
NETWORK_WINDOW_SIZE = 100
network_events = network_events.sort_values(["session_id", "pod_name", "flow_index"]).copy()
network_events["flow_window"] = (
    network_events
    .groupby(["session_id", "pod_name", "namespace", "label"])
    .cumcount()
    // NETWORK_WINDOW_SIZE
).astype(int)

agg = (
    network_events
    .groupby(metadata_cols)
    .agg(
        flow_count=("flow_index", "count"),
        unique_src_ports=("src_port", "nunique"),
        unique_dst_ports=("dst_port", "nunique"),
        reply_ratio=("is_reply", "mean"),
        **{f"{col}_count": (col, "sum") for col in count_cols}
    )
    .reset_index()
)

for base in ["EGRESS", "INGRESS"]:
    col = f"dir_{base}_count"
    if col not in agg:
        agg[col] = 0

agg["egress_ratio"] = agg["dir_EGRESS_count"] / agg["flow_count"].clip(lower=1)
agg["ingress_ratio"] = agg["dir_INGRESS_count"] / agg["flow_count"].clip(lower=1)
agg["syn_ack_ratio"] = agg["flag_SYN_count"] / agg["flag_ACK_count"].replace(0, np.nan)
agg["syn_ack_ratio"] = agg["syn_ack_ratio"].fillna(0)

network_dataset = agg.fillna(0)

X_flow = network_dataset.drop(columns=metadata_cols)
y_flow = network_dataset["scenario_label"]

variance = X_flow.var()
non_zero_cols = variance[variance > 0].index.tolist()
X_flow = X_flow[non_zero_cols]

print(f"Network rows after per-pod flow-window aggregation: {len(network_dataset):,}")
print(f"Window size: {NETWORK_WINDOW_SIZE} flows")
print(f"Features after removing zero-variance: {len(non_zero_cols)}")
print(f"Labels: {y_flow.value_counts().sort_index().to_dict()}")
print(f"Duplicate feature rows: {X_flow.duplicated().sum():,}")

k = min(20, len(non_zero_cols))
selector = SelectKBest(chi2, k=k)
X_flow_selected = selector.fit_transform(X_flow.abs(), y_flow)

selected_features = X_flow.columns[selector.get_support()].tolist()
scores = selector.scores_[selector.get_support()]

print(f"\nSelected top {k} network features:")
for feat, score in sorted(zip(selected_features, scores), key=lambda x: x[1], reverse=True):
    print(f"  {feat}: {score:.2f}")

plt.figure(figsize=(10, 6))
sorted_idx = np.argsort(scores)
plt.barh([selected_features[i] for i in sorted_idx], np.log1p(scores[sorted_idx]))
plt.xlabel("Log Chi-Square Score")
plt.ylabel("Features")
plt.tight_layout()
plt.savefig("output/network_feature_importance.png", dpi=150)
plt.show()

## 5. Split Sanity Check

In [ ]:
def grouped_split_summary(df, name):
    group_col = "session_id" if df["session_id"].nunique() > 1 else "pod_name"
    y = df["scenario_label"] if "scenario_label" in df.columns else df["label"]
    groups = df[group_col]

    splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
    train_idx, test_idx = next(splitter.split(df, y, groups=groups))

    print(f"=== {name} ===")
    print(f"Group column: {group_col}")
    print(f"Train: {len(train_idx):,} | Test: {len(test_idx):,}")
    print(f"Train labels: {Counter(y.iloc[train_idx])}")
    print(f"Test labels : {Counter(y.iloc[test_idx])}")

grouped_split_summary(syscall_dataset, "Syscall Dataset")
grouped_split_summary(network_dataset, "Network Flow Dataset")

## 6. Save Datasets

In [ ]:
Path("dataset").mkdir(exist_ok=True)
Path("output").mkdir(exist_ok=True)

syscall_dataset.to_csv("dataset/syscall_dataset.csv", index=False)
print(f"Saved syscall_dataset.csv ({len(syscall_dataset):,} rows)")

flow_df_selected = network_dataset[metadata_cols].copy()
flow_df_selected[selected_features] = pd.DataFrame(
    X_flow_selected,
    columns=selected_features,
    index=network_dataset.index,
)
flow_df_selected.to_csv("dataset/network_flow_dataset.csv", index=False)
print(f"Saved network_flow_dataset.csv ({len(flow_df_selected):,} rows)")

joblib.dump(selector, "dataset/feature_selector.pkl")
joblib.dump(selected_features, "dataset/selected_features.pkl")

print("\n=== Summary ===")
print(f"Syscall: {len(syscall_dataset):,} 5-grams | Labels: {Counter(syscall_dataset['scenario_label'])}")
print(f"Network: {len(flow_df_selected):,} flow-window rows | Labels: {Counter(flow_df_selected['scenario_label'])}")
print(f"Selected network features: {selected_features}")

## 7. Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

syscall_counts = syscall_dataset["scenario_label"].map(LABEL_NAMES).value_counts()
axes[0].barh(syscall_counts.index, syscall_counts.values)
axes[0].set_title("Syscall Dataset")
axes[0].set_xlabel("Rows")

flow_counts = flow_df_selected["scenario_label"].map(LABEL_NAMES).value_counts()
axes[1].barh(flow_counts.index, flow_counts.values)
axes[1].set_title("Network Flow Dataset")
axes[1].set_xlabel("Rows")

plt.tight_layout()
plt.savefig("output/label_distribution.png", dpi=150)
plt.show()

print("Preprocessing complete.")